# Black Jack V1

This bit uses First-visit MC policy to estimate $V \approx v_\pi$  

__Algo:__  
* I use $\epsilon-\text{greedy}$
* state $\in$ {dealer card, count, usable ace}
* action $\in$ {hit, stick}
* Infinite deck is assume

_process:__  
1. two cards to dealer (1 recorded as card up) 
2. two cards to player (ignore card up, not relevent) 
3. If player count <= 11 hit. 
4. Play:
   1. Given state take $\epsilon - \text{greedy action}$ 
   2. Continue to action is stick or bust 
   3. Update $v_\pi$ and $q_\pi$

## Updating $\pi$ and $q_\pi(s,a)$  

Run a simulation to get the following:  
1. final state: (dealer show, count, usable ace) e.g., (4, 22, 0)
2. final reward $\in (-1, 0, 1)$ , e.g.,  -1
3. Final Hand in order, e.g. [9, 1, 10, 2]
   
Evaluate backwards: $G = \gamma G + R_{t+1}$  

Process:  given a final hand ($hand$) for $len(hand) - 2$:  
Initialize: $G = 0, \gamma = 0.9$  
1. State: (4, 20, 0), action = "hit", R = -1
   * $G = \gamma(G) + -1 = 0 - 1 = -1$ 
   * increment $n_{q_\pi}((4,20,0)$ by 1 
   * Update $q_\pi((4,20,0), \text{hit}) = (q_\pi(4,20,0), \text{hit} + G) / n $
   * Update policy $\pi(a|(4,20,0)):
     * Let default value of policy be 0 (for early runs where it is not sampled)  
     * $\pi(a|(4,20,0)) = \arg\max_a(q_\pi(s = (4,20,0), A = a)$
     * set $q_\pi((4,20,0), \text{hit}) = \frac{q_\pi((4,20,0), \text{hit}) + G}{n} $
2. State: (4, 10, 1), action = "hit", R = 0 
   * $G = \gamma(G) + 0 = .9*-1 = -.9$   
   * increment $n_{q_\pi}((4,10,1)$ by 1 
   * Update $q_\pi((4,10,1), \text{hit}) = (q_\pi(4,10,1), \text{hit} + G) / n $
   * Update policy $\pi(a|(4,10,1)):
     * Let default value of policy be 0 (for early runs where it is not sampled)  
     * $\pi(a|(4,10,1)) = \arg\max_a(q_\pi(s = (4,10,1), A = a)$
     * set $q_\pi((4,10,1), \text{hit}) = \frac{q_\pi((4,10,1), \text{hit}) + G}{n} $

In [ ]:
import numpy as np
from numpy import random

In [172]:
policy = {} # a dictionary, key:{dealer card, count, usable ace} value:{hist, stick}
q = {} #The action state value: a dictionary, key:{(dealer card, count, usable ace), action} value:value (G)
q_n = {} # number of times state action pair was touched. same key as above.  value = n touched

In [354]:
def count(cards):
    '''
    Calculate count of hand.
    Returns a tuple (count, usable_ace) 
    usable_ace is binary, 1 if ace is usable Can be counted as 1 to avoid going bust on next hit.  
    There can be only one usable ace.  
    The count is the value of the hand with ace counted as 11 if usable ace or 1 if that would cause bust.  
    '''
    nAces = 0
    usable_ace = 0
    count = 0
    for card in cards:
        if card == 1:
            nAces += 1 
        count += card

    if nAces > 0 and count + 10 <= 21:
        count += 10 
        usable_ace = 1

    return([count, usable_ace])

def drawCard(n=1):
    return list(random.choice([1,2,3,4,5,6,7,8,9,10,10,10,10],size = n ))


In [355]:
def getEpsGreedyAction(policy, state, epsilon = .1):
    '''
    Get the epsilon greedy action given the state.  
    policy: the policy dictionary
    state: current state.  
    epsilon: parameter for epsilon - greedy alog.  Set to 1 for full greedy.
    '''
    action = None
    if random.rand() <= epsilon:
        action = random.choice(['stick', 'hit'])
    else:
        if state in policy:
            action = policy[state]
        else:
            action = random.choice(['stick', 'hit'])

    return action

def scoreGame(dealer_hand, player_hand):
    ''' 
    score a Game sim.  
    returns -1, 0, 1 when the player loses, draws or wins respectively
    '''
    dealer_count = count(dealer_hand)[0]
    player_count = count(player_hand)[0]

    # player busts
    if player_count > 21:
        return -1

    # dealer busts
    if dealer_count > 21:
        return 1

    # neither busts
    if player_count > dealer_count:
        return 1
    elif player_count < dealer_count:
        return -1
    else:
        return 0

def resetBoard():  
    policy = {} # a dictionary, key:{dealer card, count, usable ace} value:{hist, stick}
    q = {} #The action state value: a dictionary, key:{(dealer card, count, usable ace), action} value:value (G)
    q_n = {} # number of times state action pair was touched. same key as above.  value = n touched
    

In [ ]:
def playGame(policy, epsilon = .1):
    '''
    Run the MC sim for a single game
    '''
    dealer_hand = drawCard(2) 
    dealer_show = dealer_hand[1]
    player_hand = drawCard(2)
    
    state = tuple([dealer_show] + count(player_hand))
    action = getEpsGreedyAction(policy, state, epsilon)

    episode = [(state, action)]
    while action == 'hit' and state[1] <= 21:
        player_hand += drawCard(1)
        state = tuple([dealer_show] + count(player_hand))
        
        if state[1] > 21:
            break
        action = getEpsGreedyAction(policy, state, epsilon)
        episode.append((state, action))

    # Play dealer
    if state[1] <= 21:
        dealerCount = count(dealer_hand)
        while dealerCount[0] <= 17:
            dealer_hand += drawCard(1)
            dealerCount = count(dealer_hand)

    playerResult = scoreGame(dealer_hand=dealer_hand, player_hand=player_hand)

    res = dict(
        episode = episode,
        dealer_show = dealer_show,
        playerResult = playerResult
    )
    return(res)

In [357]:
def updateBoard(game_result, q, q_n, policy, gamma = 1):
    '''
    Function updates the q value for this state/action pair and state policy (dictionaries)   
    Updates are from the played game as defined by:
    state: final state of the game 
    action: last action of the game that resulted in the state (hit or stick)
    reward: reward (-1, 0, 1) 
    hand: the final hand of the game 
    gamma: discount used to calculate G = gamma*G + R_t+1.
    '''

    episode = game_result['episode']
    dealer_show = game_result["dealer_show"] 
    n_episode = len(episode)

    G = 0
    for i in range(n_episode):
        state = episode[n_episode - i - 1][0]
        action = episode[n_episode - i - 1][1]
        reward = game_result['playerResult'] if i == 0 else 0

        G = gamma * G + reward

        # increment n for this state, action pair
        q_n[(state, action)] = q_n.get((state, action), 0) + 1

        # Update q value for this state action pair
        old_q = q.get((state, action), 0)
        q[(state, action)] = old_q + (G - old_q) / q_n[(state, action)]

        # Update the policy
        hit_val = q.get((state, 'hit'), 0)
        stick_val = q.get((state, 'stick'), 0)
        policy[state] = 'hit' if hit_val >= stick_val else 'stick'

    return(q, q_n, policy)

    

In [406]:
q_n[(state, action)]

KeyError: ((7, 12, 1), 'stick')

In [404]:
gameResult = playGame(policy={})
gameResult

{'episode': [((7, 13, 0), 'hit'), ((7, 18, 0), 'stick')],
 'dealer_show': 7,
 'playerResult': 1}

In [405]:
updateBoard(gameResult, q, q_n, policy, gamma = 1)

({((10, 19, 0), 'hit'): -0.0003785011355034065,
  ((10, 20, 0), 'hit'): -0.0002702702702702703,
  ((5, 21, 0), 'hit'): -0.0017482517482517483,
  ((5, 11, 0), 'hit'): -0.0042654028436018955,
  ((9, 21, 0), 'hit'): -0.0016806722689075631,
  ((9, 18, 0), 'hit'): -0.001594896331738437,
  ((9, 18, 1), 'hit'): -0.011250000000000001,
  ((7, 17, 0), 'hit'): -0.0014992503748125937,
  ((10, 18, 0), 'hit'): -0.0003642250101173614,
  ((7, 15, 0), 'hit'): -0.0014150943396226416,
  ((6, 14, 0), 'hit'): -0.001601423487544484,
  ((6, 11, 0), 'hit'): -0.004186046511627907,
  ((1, 20, 0), 'hit'): -0.000992282249173098,
  ((4, 18, 0), 'hit'): -0.001466275659824047,
  ((4, 11, 0), 'hit'): -0.0033088235294117647,
  ((6, 16, 0), 'hit'): -0.001490066225165563,
  ((4, 10, 0), 'hit'): -0.0038135593220338985,
  ((4, 16, 0), 'hit'): -0.0015,
  ((4, 21, 1), 'hit'): -0.003474903474903475,
  ((7, 20, 0), 'hit'): -0.001091703056768559,
  ((7, 13, 0), 'hit'): 0.00017064846416382242,
  ((7, 5, 0), 'hit'): -0.013017857

In [358]:
def runSim(policy = {}, q = {}, q_n = {}, gamma = .9, epsilon = .1, n_cycles = 1_100_000):
    '''
    Run the simulation.  If I save policy, q and q_n I can pass them in and continue the simulation.
    '''
    for _ in range(n_cycles):
        gameResult = playGame(policy, epsilon)   
        updateBoard(gameResult, q, q_n, policy, gamma)

    res = dict(
        policy = policy,
        q = q,
        q_n = q_n
    )
    return(res)

In [378]:
x = runSim()

In [ ]:
q = x['q']
policy = x['policy']


In [381]:
import pandas as pd

rows = []

for (state, action), value in q.items():
    dealer, player_sum, usable_ace = state
    rows.append({
        'dealer': dealer,
        'player_sum': player_sum,
        'usable_ace': usable_ace,
        'action': action,
        'q_value': value
    })

df = pd.DataFrame(rows)

In [392]:
import pandas as pd

rows = []

for state, action in policy.items():
    dealer, player_sum, usable_ace = state
    rows.append({
        'dealer': dealer,
        'player_sum': player_sum,
        'usable_ace': usable_ace,
        'action': action
    })

df_policy = pd.DataFrame(rows)

In [395]:
df_policy.query('player_sum == 21')

,dealer,player_sum,usable_ace,action
2,5,21,0,stick
4,9,21,0,stick
18,4,21,1,stick
26,8,21,0,stick
46,2,21,0,stick
49,10,21,0,stick
62,2,21,1,stick
67,10,21,1,stick
87,3,21,0,stick
104,7,21,0,stick


In [396]:
df_policy.value_counts('action')

action
stick    280
Name: count, dtype: int64

In [401]:
df.action.value_counts()

action
hit    280
Name: count, dtype: int64

In [385]:
df.query('player_sum == 21').sort_values('q_value')

,dealer,player_sum,usable_ace,action,q_value
155,1,21,1,hit,-0.003673
135,8,21,1,hit,-0.003629
105,7,21,1,hit,-0.003529
62,2,21,1,hit,-0.003516
18,4,21,1,hit,-0.003475
112,9,21,1,hit,-0.003409
162,6,21,1,hit,-0.003358
123,5,21,1,hit,-0.003164
164,3,21,1,hit,-0.003140
46,2,21,0,hit,-0.001812


In [371]:
for k, v in q.items():
    if k[1] == 'stick':
        print(k,v)

In [362]:
sorted_q = sorted(
    q.items(),
    key=lambda x: (x[0][0][1], x[0][0][2])
)

sorted_q

[(((8, 4, 0), 'hit'), -0.05),
 (((1, 4, 0), 'hit'), -0.031153846153846157),
 (((9, 4, 0), 'hit'), -0.075),
 (((5, 4, 0), 'hit'), -0.035217391304347825),
 (((6, 4, 0), 'hit'), -0.0405),
 (((10, 4, 0), 'hit'), -0.008571428571428572),
 (((7, 4, 0), 'hit'), -0.028125),
 (((2, 4, 0), 'hit'), -0.03214285714285715),
 (((3, 4, 0), 'hit'), -0.0391304347826087),
 (((4, 4, 0), 'hit'), -0.018452812500000006),
 (((7, 5, 0), 'hit'), -0.013017857142857145),
 (((2, 5, 0), 'hit'), -0.018367346938775512),
 (((8, 5, 0), 'hit'), -0.01723404255319149),
 (((10, 5, 0), 'hit'), -0.00400990099009901),
 (((4, 5, 0), 'hit'), -0.015789473684210527),
 (((6, 5, 0), 'hit'), -0.012857142857142859),
 (((9, 5, 0), 'hit'), -0.013728813559322034),
 (((1, 5, 0), 'hit'), -0.01956521739130435),
 (((3, 5, 0), 'hit'), -0.014210526315789474),
 (((5, 5, 0), 'hit'), -0.012568965517241381),
 (((2, 6, 0), 'hit'), -0.01173913043478261),
 (((6, 6, 0), 'hit'), -0.010253164556962025),
 (((5, 6, 0), 'hit'), -0.009592105263157897),
 (((

In [366]:
for k, v in policy.items(): 
    if v != 'stick':
        print(k,v)